In [0]:
%sql
CREATE TEMP VIEW employees AS
SELECT * FROM VALUES
  (1, 'CEO', NULL),
  (2, 'CTO', 1),
  (3, 'CFO', 1),
  (4, 'Engineering Manager', 2),
  (5, 'Finance Manager', 3),
  (6, 'Software Engineer', 4),
  (7, 'Accountant', 5)
AS employees(employee_id, employee_name, manager_id);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS raw_data;
CREATE SCHEMA IF NOT EXISTS hub_data;
CREATE SCHEMA IF NOT EXISTS mart_data;

In [0]:
%sql
CREATE TABLE raw_schema.Customer (
    CustomerID INT,
    CustomerName STRING,
    LastName STRING,
    Country STRING,
    Age INT,
    Phone STRING
);
ALTER TABLE raw_schema.Customer ADD CONSTRAINT ageRangeCheck CHECK (Age > 0 AND Age <120);




In [0]:
%sql
CREATE TABLE hub_data.Customer (
    CustomerID INT,
    CustomerName STRING,
    LastName STRING,
    Country STRING,
    Age INT,
    Phone STRING
);

In [0]:
%sql
INSERT INTO raw_schema.Customer VALUES
  (12, 'John Dicking', 'Dicking', 'USA', 15, "0")

In [0]:
%sql
INSERT INTO hub_data.Customer
  SELECT * 
  FROM raw_schema.Customer WHERE Age > 10

In [0]:
%sql
SELECT * FROM raw_schema.customer;

In [0]:
from pyspark.sql.functions import expr
from datetime import datetime

# Create demo DataFrame
data = spark.range(0, 10000).selectExpr(
    "id as order_id",
    "CAST(id % 50 AS STRING) AS user_id",  # 50 users
    "date_sub(current_date(), CAST(id % 10 AS INT)) AS event_date",
    "CAST(rand() * 100 AS DECIMAL(10,2)) AS amount"
)


data.createOrReplaceTempView("tmp_sales")

In [0]:
%sql
-- Not working on serverless, run manually
-- dbutils.notebook.run("./dataset-view-setup",timeout_seconds=60)
CREATE OR REPLACE TEMP VIEW sales_demo AS
SELECT * FROM VALUES
  ('2025-07-01', 'Apples', 10, 2.5, 'North'),
  ('2025-07-01', 'Oranges', 5, 3.0, 'North'),
  ('2025-07-02', 'Apples', 8, 2.5, 'South'),
  ('2025-07-02', 'Bananas', 15, 1.2, Null),
  ('2025-07-03', 'Oranges', 7, 3.0, 'East')
AS sales(date, product, quantity, price_per_unit, region);

-- product categories table
CREATE OR REPLACE TEMP VIEW product_categories AS
SELECT * FROM VALUES
  ('Apples', 'Fruit'),
  ('Oranges', 'Fruit'),
  ('Tomatoes', 'Vegetable')
AS categories(product, category);

-- Sales extra (for union)
CREATE OR REPLACE TEMP VIEW sales_extra AS
SELECT * FROM VALUES
  ('2025-07-04', 'Apples', 6, 2.5, 'North'),
  ('2025-07-04', 'Oranges', 9, 3.0, 'West'),
  ('2025-07-03', 'Oranges', 7, 3.0, 'East')
AS sales(date, product, quantity, price_per_unit, region);

-- Customers table
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Alice'),
  (2, 'Bob'),
  (3, 'Charlie'),
  (4, 'Diana')
AS customers(customer_id, name);

-- Orders table
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (101, 1, '2023-01-10'),
  (102, 1, '2023-03-15'),
  (103, 1, '2023-06-20'),
  (104, 2, '2023-02-12'),
  (105, 3, '2023-04-25'),
  (106, 3, '2023-07-30')
AS orders(order_id, customer_id, order_date);

In [0]:
%sql
SELECT * FROM orders o LEFT OUTER JOIN customers c ON o.customer_id = c.customer_id;

In [0]:
%sql
SELECT s.*, c.category
FROM sales_demo s
LEFT JOIN product_categories c
  ON s.product = c.product;